In [1]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# Locate repo root and add project/ to path
REPO_ROOT = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists())
PROJECT_DIR = REPO_ROOT / 'project'
sys.path.insert(0, str(PROJECT_DIR))

load_dotenv(REPO_ROOT / '.env')

TOOLBENCH_DIR = Path(os.environ.get('TOOLBENCH_DIR', str(REPO_ROOT / 'toolbench_data')))
print(f"Repo:      {REPO_ROOT}")
print(f"Toolbench: {TOOLBENCH_DIR}")

Repo:      /Users/michaelserrano/Documents/GitHub/mmai
Toolbench: /Users/michaelserrano/Documents/GitHub/mmai/toolbench_data


In [3]:
import numpy as np, json
from data.load_toolbench import load_api_corpus, load_eval_examples
from models.embeddings import get_embeddings
from retrieval.retriever import build_faiss_index, retrieve_top_k
from evaluation.metrics import evaluate_batch

corpus = load_api_corpus(TOOLBENCH_DIR / "toolenv" / "tools")
corpus_matrix = np.load(PROJECT_DIR / "corpus_embeddings.npy")
with open(PROJECT_DIR / "api_names.json") as f:
    api_names = json.load(f)
name_to_idx = {name: i for i, name in enumerate(api_names)}
evals = load_eval_examples(TOOLBENCH_DIR / "toolllama_G123_dfs_eval.json")
print(f"Corpus: {len(corpus)} APIs | Eval: {len(evals)} examples")

Corpus: 49937 APIs | Eval: 747 examples


In [4]:
index = build_faiss_index(corpus_matrix)
print(f"\u2713 FAISS index ready: {index.ntotal} vectors")

✓ FAISS index ready: 49937 vectors


In [5]:
from tqdm import tqdm
queries = [e["user_query"] for e in evals]
query_embeddings = get_embeddings(queries)
print(f"\u2713 Query embeddings: {query_embeddings.shape}")

✓ Query embeddings: (747, 1536)


In [6]:
K = 10
all_retrieved = []
all_gt_indices = []

for i, ex in enumerate(evals):
    top_k_indices = retrieve_top_k(query_embeddings[i], index, k=K)
    all_retrieved.append(top_k_indices)
    gt_indices = [name_to_idx[name] for name in ex["ground_truth_apis"] if name in name_to_idx]
    all_gt_indices.append(gt_indices)

results = evaluate_batch(all_retrieved, all_gt_indices, ks=[1, 5, 10])
print("\n=== Baseline Results (Full Corpus / Random Negatives) ===")
for metric, score in results.items():
    print(f"  {metric}: {score:.4f}")


=== Baseline Results (Full Corpus / Random Negatives) ===
  recall@1: 0.2232
  recall@5: 0.4212
  recall@10: 0.5062
  mrr: 0.4346


In [7]:
import json
with open(PROJECT_DIR / 'results_baseline.json', 'w') as f:
    json.dump({'condition': 'random_full_corpus', 'results': results}, f, indent=2)
print("✓ Saved results_baseline.json")

✓ Saved results_baseline.json
